In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, gc, itertools
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

In [ ]:
# heartbeat utilities

HEARTBEAT_SECS = 600

def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

In [ ]:
# paths

DATASET_PATH    = "/content/drive/MyDrive/298/Jena/Data/jena_climate_2009_2016.csv"
BASE_PROJ_DIR   = "/content/drive/MyDrive/298/Jena/proj_v2"

BASE_MODEL_DIR      = os.path.join(BASE_PROJ_DIR, "baseModel")
BASE_STUDENTS_DIR   = os.path.join(BASE_MODEL_DIR, "baseStudents")
BASE_LOGS_DIR       = os.path.join(BASE_MODEL_DIR, "logs")

FGSM_DIR            = os.path.join(BASE_PROJ_DIR, "fgsm")
FGSM_ADV_DIR        = os.path.join(FGSM_DIR, "advStudents")
FGSM_RESULTS_DIR    = os.path.join(FGSM_DIR, "results")
FGSM_LOGS_DIR       = os.path.join(FGSM_RESULTS_DIR, "train_logs")

BIM_DIR             = os.path.join(BASE_PROJ_DIR, "bim")
BIM_ADV_DIR         = os.path.join(BIM_DIR, "advStudents")
BIM_RESULTS_DIR     = os.path.join(BIM_DIR, "results")
BIM_LOGS_DIR        = os.path.join(BIM_RESULTS_DIR, "train_logs")

PGD_DIR             = os.path.join(BASE_PROJ_DIR, "pgd")
PGD_ADV_DIR         = os.path.join(PGD_DIR, "advStudents")
PGD_RESULTS_DIR     = os.path.join(PGD_DIR, "results")
PGD_LOGS_DIR        = os.path.join(PGD_RESULTS_DIR, "train_logs")

for d in [
    BASE_PROJ_DIR, BASE_MODEL_DIR, BASE_STUDENTS_DIR, BASE_LOGS_DIR,
    FGSM_DIR, FGSM_ADV_DIR, FGSM_RESULTS_DIR, FGSM_LOGS_DIR,
    BIM_DIR,  BIM_ADV_DIR,  BIM_RESULTS_DIR,  BIM_LOGS_DIR,
    PGD_DIR,  PGD_ADV_DIR,  PGD_RESULTS_DIR,  PGD_LOGS_DIR,
]:
    os.makedirs(d, exist_ok=True)

FGSM_RUNS_CSV   = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_runs.csv")
FGSM_STATS_CSV  = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_stats.csv")
FGSM_WEIGHT_DIVERSITY_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_weight_diversity.csv")
FGSM_PRED_DIVERSITY_CSV   = os.path.join(FGSM_RESULTS_DIR, "fgsm_prediction_diversity.csv")
FGSM_XFER_MATRIX_CSV      = os.path.join(FGSM_RESULTS_DIR, "fgsm_transfer_matrix.csv")

BIM_RUNS_CSV    = os.path.join(BIM_RESULTS_DIR, "bim_morphence_runs.csv")
BIM_STATS_CSV   = os.path.join(BIM_RESULTS_DIR, "bim_morphence_stats.csv")
BIM_WEIGHT_DIVERSITY_CSV  = os.path.join(BIM_RESULTS_DIR, "bim_weight_diversity.csv")
BIM_PRED_DIVERSITY_CSV    = os.path.join(BIM_RESULTS_DIR, "bim_prediction_diversity.csv")
BIM_XFER_MATRIX_CSV       = os.path.join(BIM_RESULTS_DIR, "bim_transfer_matrix.csv")

PGD_RUNS_CSV    = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_runs.csv")
PGD_STATS_CSV   = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_stats.csv")
PGD_WEIGHT_DIVERSITY_CSV  = os.path.join(PGD_RESULTS_DIR, "pgd_weight_diversity.csv")
PGD_PRED_DIVERSITY_CSV    = os.path.join(PGD_RESULTS_DIR, "pgd_prediction_diversity.csv")
PGD_XFER_MATRIX_CSV       = os.path.join(PGD_RESULTS_DIR, "pgd_transfer_matrix.csv")

BASE_TRAIN_HISTORY_CSV = os.path.join(BASE_LOGS_DIR, "jena_base_train_history.csv")

print("proj_v2 base dir:", BASE_PROJ_DIR)

proj_v2 base dir: /content/drive/MyDrive/298/Jena/proj_v2


In [ ]:
# global configs

import threading

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

LOOKBACK   = 24
TARGET_COL = "t_degc"
BATCH      = 64
N_STUDENTS = 4

EPS_LIST = [0.1, 0.2, 0.3, 0.5]
FGSM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
BIM_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
PGD_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]

BIM_ITERS  = 10
PGD_ITERS  = 10

FGSM_EPOCHS = 5
BIM_EPOCHS  = 5
PGD_EPOCHS  = 5

LR_BASE    = 1e-3
LR_STUDENT = 1e-3

Device: cuda


In [ ]:
# data prep

df = pd.read_csv(DATASET_PATH)
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.set_index("Date Time").sort_index()
df = df[~df.index.duplicated(keep="first")]

df = df.resample("h").ffill()
if df.head(1).isna().any(axis=None):
    df = df.bfill(limit=1)

col_map = {
    "p (mbar)": "p_mbar",
    "T (degC)": "t_degc",
    "Tpot (K)": "tpot_k",
    "rh (%)":   "rh_pct",
    "wv (m/s)": "wv_ms",
}
df_sel = df[list(col_map.keys())].rename(columns=col_map)

print("After hourly resample + ffill/bfill:")
print("Shape:", df_sel.shape)
print("Columns:", df_sel.columns.tolist())

split_idx = int(0.8 * len(df_sel))
train_df, test_df = df_sel.iloc[:split_idx].copy(), df_sel.iloc[split_idx:].copy()

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(train_df),
    index=train_df.index,
    columns=train_df.columns,
)
test_scaled = pd.DataFrame(
    scaler.transform(test_df),
    index=test_df.index,
    columns=test_df.columns,
)

def make_windows(frame, lookback=24, target_col="t_degc"):
    vals = frame.values
    tgt_idx = frame.columns.get_loc(target_col)
    X, y, idx = [], [], []
    for i in range(lookback, len(vals)):
        X.append(vals[i-lookback:i])
        y.append(vals[i][tgt_idx])
        idx.append(frame.index[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(idx)

X_train, y_train, t_train = make_windows(train_scaled, LOOKBACK, TARGET_COL)
X_test,  y_test,  t_test  = make_windows(test_scaled,  LOOKBACK, TARGET_COL)

print("\nWindowed tensors")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

N_tr = len(X_train)
n_tr_core = int(0.9 * N_tr)

X_tr_core, y_tr_core = X_train[:n_tr_core], y_train[:n_tr_core]
X_val,     y_val     = X_train[n_tr_core:], y_train[n_tr_core:]

print("\nTrain/Val/Test windowed splits:")
print("  train_core:", X_tr_core.shape, y_tr_core.shape)
print("  val       :", X_val.shape,     y_val.shape)
print("  test      :", X_test.shape,    y_test.shape)

class TSForecastDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_dl = DataLoader(
    TSForecastDataset(X_tr_core, y_tr_core),
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
)

val_dl = DataLoader(
    TSForecastDataset(X_val, y_val),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
)

test_dl = DataLoader(
    TSForecastDataset(X_test, y_test),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
)

After hourly resample + ffill/bfill:
Shape: (70129, 5)
Columns: ['p_mbar', 't_degc', 'tpot_k', 'rh_pct', 'wv_ms']

Windowed tensors
X_train: (56079, 24, 5) y_train: (56079,)
X_test:  (14002, 24, 5) y_test:  (14002,)

Train/Val/Test windowed splits:
  train_core: (50471, 24, 5) (50471,)
  val       : (5608, 24, 5) (5608,)
  test      : (14002, 24, 5) (14002,)


In [ ]:
# model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerRegressor(nn.Module):
    def __init__(self, in_feats=5, d_model=128, nhead=4,
                 num_layers=4, dim_ff=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(in_feats, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.posenc  = PositionalEncoding(d_model)
        self.head    = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        z  = self.in_proj(x)
        z  = self.posenc(z)
        z  = self.encoder(z)
        zT = z[:, -1, :]
        return self.head(zT)

mse = nn.MSELoss()
def rmse_loss(pred, true): return torch.sqrt(mse(pred, true))

def quantile_loss(true, pred, q=0.5):
    e = true - pred
    return torch.mean(torch.maximum(q*e, (q-1)*e))

In [ ]:
# base training

def train_base(model, train_dl, val_dl, epochs=20, lr=1e-3, q=0.5):
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr)
    history = []

    base_best_ckpt = os.path.join(BASE_MODEL_DIR, "jena_base_best.pth")

    hb_flag, hb_thread = start_heartbeat("base_train")
    start_time = time.time()

    best_val_loss = float("inf")

    try:
        for ep in range(1, epochs + 1):

            # train epoch
            model.train()
            tot_train = cnt_train = 0.0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred   = model(xb)
                loss   = rmse_loss(pred, yb) + quantile_loss(yb, pred, q)

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot_train += loss.item() * bs
                cnt_train += bs

            train_loss = tot_train / max(1, cnt_train)

            # val epoch
            model.eval()
            tot_val = cnt_val = 0.0
            with torch.no_grad():
                for xb, yb in val_dl:
                    xb, yb = xb.to(device), yb.to(device)
                    pred   = model(xb)
                    v_loss = rmse_loss(pred, yb) + quantile_loss(yb, pred, q)
                    bs = xb.size(0)
                    tot_val += v_loss.item() * bs
                    cnt_val += bs

            val_loss = tot_val / max(1, cnt_val)

            ep_mins    = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            print(
                f"[Base] Epoch {ep:02d} | train={train_loss:.4f} | "
                f"val={val_loss:.4f} | ep_time={ep_mins:.2f} min | total={total_mins:.2f} min"
            )

            history.append({
                "epoch": ep,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            })

            # per epoch checkpoint
            ep_ckpt = os.path.join(BASE_MODEL_DIR, f"jena_base_epoch_{ep:02d}.pth")
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

            # track best val loss
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.to("cpu").state_dict(), base_best_ckpt)
                model.to(device)
                print("  [Base] Updated best checkpoint (val):", base_best_ckpt)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # save history
    df_hist = pd.DataFrame(history)
    df_hist.to_csv(BASE_TRAIN_HISTORY_CSV, index=False)
    print("Saved base train history to:", BASE_TRAIN_HISTORY_CSV)

    print("Final best base checkpoint (based on val):", base_best_ckpt)
    return base_best_ckpt

In [ ]:
# train/load base model

base_best_ckpt = os.path.join(BASE_MODEL_DIR, "jena_base_best.pth")

if os.path.exists(base_best_ckpt):
    print("Loading existing best base model from:", base_best_ckpt)
else:
    print("Training new base model...")
    base_init = TransformerRegressor(in_feats=X_train.shape[-1])
    base_best_ckpt = train_base(
        base_init,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=20,
        lr=LR_BASE,
    )

# load best checkpoint to base
base = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
base.load_state_dict(torch.load(base_best_ckpt, map_location=device))
base.eval()
print("Base model loaded from best checkpoint:", base_best_ckpt)

Training new base model...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[Base] Epoch 01 | train=0.0899 | val=0.0437 | ep_time=0.19 min | total=0.19 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v2/baseModel/jena_base_best.pth
[Base] Epoch 02 | train=0.0317 | val=0.0175 | ep_time=0.17 min | total=0.36 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v2/baseModel/jena_base_best.pth
[Base] Epoch 03 | train=0.0261 | val=0.0409 | ep_time=0.17 min | total=0.53 min
[Base] Epoch 04 | train=0.0250 | val=0.0156 | ep_time=0.17 min | total=0.70 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v2/baseModel/jena_base_best.pth
[Base] Epoch 05 | train=0.0230 | val=0.0221 | ep_time=0.17 min | total=0.87 min
[Base] Epoch 06 | train=0.0227 | val=0.0171 | ep_time=0.17 min | total=1.04 min
[Base] Epoch 07 | train=0.0222 | val=0.0152 | ep_time=0.17 min | total=1.21 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v2/baseModel/jena_base_best.pth
[Bas

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [ ]:
# load base model

base_best_ckpt = os.path.join(BASE_MODEL_DIR, "jena_base_best.pth")

# load best checkpoint to base
base = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
base.load_state_dict(torch.load(base_best_ckpt, map_location=device))
base.eval()
print("Base model loaded from best checkpoint:", base_best_ckpt)

Base model loaded from best checkpoint: /content/drive/MyDrive/298/Jena/proj_v2/baseModel/jena_base_best.pth


In [ ]:
# generate base students

os.makedirs(BASE_STUDENTS_DIR, exist_ok=True)

def save_student(model_state, idx, noise_std=1e-3):
    st = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
    st.load_state_dict(model_state)
    with torch.no_grad():
        for _, p in st.named_parameters():
            p.add_(noise_std * torch.randn_like(p))
    out_path = os.path.join(BASE_STUDENTS_DIR, f"student_base_{idx:02d}.pth")
    torch.save(st.state_dict(), out_path)
    print("Saved base student:", out_path)
    return out_path

base_cpu = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
base_cpu.load_state_dict(torch.load(base_best_ckpt, map_location="cpu"))

existing_students = [
    f for f in os.listdir(BASE_STUDENTS_DIR) if f.endswith(".pth")
]
if len(existing_students) >= N_STUDENTS:
    print(f"Found {len(existing_students)} base students, not regenerating.")
else:
    print("Generating base students...")
    for i in range(1, N_STUDENTS + 1):
        save_student(copy.deepcopy(base_cpu.state_dict()), i, noise_std=1e-3)

base_student_paths = sorted(
    os.path.join(BASE_STUDENTS_DIR, f)
    for f in os.listdir(BASE_STUDENTS_DIR)
    if f.endswith(".pth")
)
print("\nBase student paths:")
for p in base_student_paths:
    print("  ", p)

Generating base students...
Saved base student: /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_01.pth
Saved base student: /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_02.pth
Saved base student: /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_03.pth
Saved base student: /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_04.pth

Base student paths:
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_01.pth
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_02.pth
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_03.pth
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_04.pth


In [ ]:
# attacks

# fgsm
def rfgsm(model, X, Y, eps=0.1, alpha=0.02):
    was_training = model.training
    model.eval()
    X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    X_adv.requires_grad_(True)
    loss = mse(model(X_adv), Y)
    loss.backward()
    X_adv = (X_adv + alpha * X_adv.grad.sign()).clamp(0, 1)
    if was_training:
        model.train()
    return X_adv.detach()

#bim
def bim_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=False):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)
    for _ in range(iters):
        preds = model(X_adv)
        loss  = mse(preds, Y)
        if X_adv.grad is not None:
            X_adv.grad.zero_()
        loss.backward()
        grad_sign = X_adv.grad.sign()
        X_adv     = (X_adv + alpha * grad_sign).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0, 1).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()
    return X_adv.detach()

#pgd
def pgd_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=True):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)
    for _ in range(iters):
        preds = model(X_adv)
        loss  = mse(preds, Y)
        if X_adv.grad is not None:
            X_adv.grad.zero_()
        loss.backward()
        grad_sign = X_adv.grad.sign()
        X_adv     = (X_adv + alpha * grad_sign).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0, 1).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()
    return X_adv.detach()

In [ ]:
# student adv training definitions

#fgsm
def adv_train_student_fgsm(student_path, out_dir, logs_dir,
                           out_suffix="_fgsm_adv",
                           epochs=5, lr=1e-3,
                           eps_list=None, alpha=0.02, q=0.5):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = FGSM_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"fgsm_adv_train_{student_stub}")
    start_time = time.time()

    try:
        for ep in range(1, epochs+1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = rfgsm(model, xb, yb, eps=eps, alpha=alpha)
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad(); L.backward(); opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean   += Lc.item() * bs
                tot_adv_mean+= La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch   = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(f"[FGSM train {student_stub}] "
                  f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                  f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m")

            # per-epoch checkpoint
            ep_ckpt = os.path.join(
                out_dir,
                f"{student_stub}{out_suffix}_epoch{ep:02d}.pth"
            )
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # final adv student
    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth")
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved FGSM-adv student:", out)

    # save train history
    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved FGSM train history to:", hist_path)

    return out

#bim
def adv_train_student_bim(student_path, out_dir, logs_dir,
                          out_suffix="_bim_adv",
                          epochs=5, lr=1e-3,
                          eps_list=None, iters=10, q=0.5,
                          random_start=False):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = BIM_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"bim_adv_train_{student_stub}")
    start_time = time.time()

    try:
        for ep in range(1, epochs+1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = bim_attack(
                        model, xb, yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad(); L.backward(); opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean   += Lc.item() * bs
                tot_adv_mean+= La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch   = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(f"[BIM train {student_stub}] "
                  f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                  f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m")

            ep_ckpt = os.path.join(
                out_dir,
                f"{student_stub}{out_suffix}_epoch{ep:02d}.pth"
            )
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth")
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved BIM-adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved BIM train history to:", hist_path)

    return out

#pgd
def adv_train_student_pgd(student_path, out_dir, logs_dir,
                          out_suffix="_pgd_adv",
                          epochs=5, lr=1e-3,
                          eps_list=None, iters=10, q=0.5,
                          random_start=True):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = PGD_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"pgd_adv_train_{student_stub}")
    start_time = time.time()

    try:
        for ep in range(1, epochs+1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = pgd_attack(
                        model, xb, yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad(); L.backward(); opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean   += Lc.item() * bs
                tot_adv_mean+= La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch   = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(f"[PGD train {student_stub}] "
                  f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                  f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m")

            ep_ckpt = os.path.join(
                out_dir,
                f"{student_stub}{out_suffix}_epoch{ep:02d}.pth"
            )
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth")
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved PGD-adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved PGD train history to:", hist_path)

    return out


In [ ]:
# adv training

print("\nTraining FGSM adv students")
fgsm_adv_paths = []
for sp in base_student_paths:
    out = adv_train_student_fgsm(
        sp,
        out_dir=FGSM_ADV_DIR,
        logs_dir=FGSM_LOGS_DIR,
        out_suffix="_fgsm_adv",
        epochs=FGSM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=FGSM_TRAIN_EPS_LIST,
        alpha=0.02,
    )
    fgsm_adv_paths.append(out)

print("\nTraining BIM adv students")
bim_adv_paths = []
for sp in base_student_paths:
    out = adv_train_student_bim(
        sp,
        out_dir=BIM_ADV_DIR,
        logs_dir=BIM_LOGS_DIR,
        out_suffix="_bim_adv",
        epochs=BIM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=BIM_TRAIN_EPS_LIST,
        iters=BIM_ITERS,
        random_start=False,
    )
    bim_adv_paths.append(out)

print("\nTraining PGD adv students")
pgd_adv_paths = []
for sp in base_student_paths:
    out = adv_train_student_pgd(
        sp,
        out_dir=PGD_ADV_DIR,
        logs_dir=PGD_LOGS_DIR,
        out_suffix="_pgd_adv",
        epochs=PGD_EPOCHS,
        lr=LR_STUDENT,
        eps_list=PGD_TRAIN_EPS_LIST,
        iters=PGD_ITERS,
        random_start=True,
    )
    pgd_adv_paths.append(out)


Training FGSM adv students
[FGSM train student_base_01] epoch 1/5 | clean=0.0322 | adv_mean=0.0935 | ep=1.23m | total=1.24m
[FGSM train student_base_01] epoch 2/5 | clean=0.0279 | adv_mean=0.0891 | ep=1.23m | total=2.47m
[FGSM train student_base_01] epoch 3/5 | clean=0.0264 | adv_mean=0.0879 | ep=1.25m | total=3.71m
[FGSM train student_base_01] epoch 4/5 | clean=0.0252 | adv_mean=0.0865 | ep=1.24m | total=4.95m
[FGSM train student_base_01] epoch 5/5 | clean=0.0240 | adv_mean=0.0855 | ep=1.25m | total=6.20m
Saved FGSM-adv student: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/advStudents/student_base_01_fgsm_adv.pth
Saved FGSM train history to: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/train_logs/student_base_01_fgsm_adv_train_history.csv
[FGSM train student_base_02] epoch 1/5 | clean=0.0324 | adv_mean=0.0935 | ep=1.24m | total=1.24m
[FGSM train student_base_02] epoch 2/5 | clean=0.0283 | adv_mean=0.0897 | ep=1.23m | total=2.47m
[FGSM train student_base_02] epoch 3/5 | clean=

In [ ]:
# eval helpers

def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            p  = model(xb)
            ys.append(yb.cpu().numpy())
            ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))

def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = []
            for m in models:
                preds.append(m(xb))
            preds = torch.stack(preds, dim=0).mean(dim=0)
            ys.append(yb.cpu().numpy())
            ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))

def eval_pair_attack(defender, attacker, loader, attack_fn, atk_kwargs):
    defender.eval()
    attacker.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        xa = attack_fn(attacker, xb, yb, **atk_kwargs)
        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))

def eval_random_switch_attack(models, loader, attack_fn, atk_kwargs_func):

    #atk_kwargs_func: function() - dict with eps/alpha/iters already baked in

    for m in models:
        m.eval()
    ys, ps = [], []
    n_stu = len(models)

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        atk_model = random.choice(models)
        def_model = random.choice(models)

        atk_kwargs = atk_kwargs_func()

        xa = attack_fn(atk_model, xb, yb, **atk_kwargs)
        with torch.no_grad():
            p = def_model(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))

In [ ]:
# diversity metrics
# weight + prediction

def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity(students, names, out_csv):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("Saved weight diversity to:", out_csv)
    return df_w

@torch.no_grad()
def compute_prediction_diversity(students, loader, out_csv):

    for m in students:
        m.eval()

    k = len(students)
    sum_pred  = None
    sum_pred2 = None
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)

        preds_stu = []
        for m in students:
            # each: (B, 1) -> flatten to (B,)
            p = m(xb).cpu().numpy().reshape(-1)
            preds_stu.append(p)

        # (k, B)
        preds_stu = np.stack(preds_stu, axis=0)

        # average over batch dimension -> (k,)
        preds_batch_mean = preds_stu.mean(axis=1)

        if sum_pred is None:
            sum_pred  = np.zeros_like(preds_batch_mean, dtype=np.float64)
            sum_pred2 = np.zeros_like(preds_batch_mean, dtype=np.float64)

        sum_pred  += preds_batch_mean
        sum_pred2 += preds_batch_mean ** 2
        count     += 1

    if count == 0:
        raise RuntimeError("compute_prediction_diversity: loader produced 0 batches")

    mean_pred  = sum_pred  / count          # (k,)
    mean_pred2 = sum_pred2 / count          # (k,)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "student_idx": np.arange(k),
        "mean_pred":   mean_pred,
        "var_pred":    var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("Saved prediction diversity to:", out_csv)
    return df_p

In [ ]:
# transferability matrix
# per epsilon, per attack type

def compute_transfer_matrix(
    students,
    names,
    loader,
    attack_fn,
    eps_list,
    extra_kwargs_func,
    out_csv,
    attack_label="rfgsm",
):
    rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_label}_transfer_matrix")
    start_time = time.time()

    try:
        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)
            print(f"\n[{attack_label.upper()}] Transfer matrix for eps={eps}...")
            for i, atk_name in enumerate(names):
                for j, def_name in enumerate(names):
                    atk_model = students[i]
                    def_model = students[j]
                    rmse_ij = eval_pair_attack(
                        defender=def_model,
                        attacker=atk_model,
                        loader=loader,
                        attack_fn=attack_fn,
                        atk_kwargs=atk_kwargs,
                    )
                    rows.append({
                        "attack": attack_label,
                        "epsilon": eps,
                        "attacker": atk_name,
                        "defender": def_name,
                        "rmse_adv": rmse_ij,
                    })
                    print(f"  atk={atk_name} - def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")
    finally:
        stop_heartbeat(hb_flag, hb_thread)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {attack_label} transfer matrix to:", out_csv)
    return df

In [ ]:
# reload base students
# reload adv students

# base student paths
BASE_STUDENTS_DIR = os.path.join(BASE_MODEL_DIR, "baseStudents")
base_student_paths = sorted([
    os.path.join(BASE_STUDENTS_DIR, f)
    for f in os.listdir(BASE_STUDENTS_DIR)
    if f.endswith(".pth")
])

print("Base students found:", len(base_student_paths))
for p in base_student_paths:
    print("  ", p)

# fgsm adv students
FGSM_ADV_DIR = os.path.join(FGSM_DIR, "advStudents")
fgsm_adv_paths = sorted([
    os.path.join(FGSM_ADV_DIR, f)
    for f in os.listdir(FGSM_ADV_DIR)
    if f.endswith("_fgsm_adv.pth")
])

print("\nFGSM adv students found:", len(fgsm_adv_paths))
for p in fgsm_adv_paths:
    print("  ", p)

# bim adv students
BIM_ADV_DIR = os.path.join(BIM_DIR, "advStudents")
bim_adv_paths = sorted([
    os.path.join(BIM_ADV_DIR, f)
    for f in os.listdir(BIM_ADV_DIR)
    if f.endswith("_bim_adv.pth")
])

print("\nBIM adv students found:", len(bim_adv_paths))
for p in bim_adv_paths:
    print("  ", p)

# pgd adv students
PGD_ADV_DIR = os.path.join(PGD_DIR, "advStudents")
pgd_adv_paths = sorted([
    os.path.join(PGD_ADV_DIR, f)
    for f in os.listdir(PGD_ADV_DIR)
    if f.endswith("_pgd_adv.pth")
])

print("\nPGD adv students found:", len(pgd_adv_paths))
for p in pgd_adv_paths:
    print("  ", p)



# move to device
def load_adv_students(path_list):
    students = []
    for p in path_list:
        m = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval()
        students.append(m)
    return students

# load pools
fgsm_students = load_adv_students(fgsm_adv_paths)
bim_students  = load_adv_students(bim_adv_paths)
pgd_students  = load_adv_students(pgd_adv_paths)

fgsm_student_names = [os.path.basename(p).replace(".pth", "") for p in fgsm_adv_paths]
bim_student_names  = [os.path.basename(p).replace(".pth", "") for p in bim_adv_paths]
pgd_student_names  = [os.path.basename(p).replace(".pth", "") for p in pgd_adv_paths]

print("\nLoaded adv student pools:")
print(" FGSM:", len(fgsm_students))
print(" BIM :", len(bim_students))
print(" PGD :", len(pgd_students))

Base students found: 4
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_01.pth
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_02.pth
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_03.pth
   /content/drive/MyDrive/298/Jena/proj_v2/baseModel/baseStudents/student_base_04.pth

FGSM adv students found: 4
   /content/drive/MyDrive/298/Jena/proj_v2/fgsm/advStudents/student_base_01_fgsm_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/fgsm/advStudents/student_base_02_fgsm_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/fgsm/advStudents/student_base_03_fgsm_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/fgsm/advStudents/student_base_04_fgsm_adv.pth

BIM adv students found: 4
   /content/drive/MyDrive/298/Jena/proj_v2/bim/advStudents/student_base_01_bim_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/bim/advStudents/student_base_02_bim_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/bim/a

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



Loaded adv student pools:
 FGSM: 4
 BIM : 4
 PGD : 4


In [ ]:
# build adv student model lists

def load_adv_students(adv_paths):
    models = []
    for p in adv_paths:
        m = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval()
        models.append(m)
    return models

# common base_eval
base_eval = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
base_eval.load_state_dict(torch.load(base_best_ckpt, map_location=device))
base_eval.eval()

fgsm_students = load_adv_students(sorted(fgsm_adv_paths))
bim_students  = load_adv_students(sorted(bim_adv_paths))
pgd_students  = load_adv_students(sorted(pgd_adv_paths))

fgsm_student_names = [os.path.basename(p).replace(".pth", "") for p in sorted(fgsm_adv_paths)]
bim_student_names  = [os.path.basename(p).replace(".pth", "") for p in sorted(bim_adv_paths)]
pgd_student_names  = [os.path.basename(p).replace(".pth", "") for p in sorted(pgd_adv_paths)]

print("\nLoaded adv student pools:")
print(" FGSM:", len(fgsm_students))
print(" BIM :", len(bim_students))
print(" PGD :", len(pgd_students))


Loaded adv student pools:
 FGSM: 4
 BIM : 4
 PGD : 4


In [ ]:
# 30 run eval

def run_morphence_eval(
    attack_name,
    attack_fn,
    eps_list,
    students,
    results_csv,
    stats_csv,
    extra_kwargs_func,
    n_runs=30,
):
    """
    attack_name: "rfgsm", "bim", "pgd"
    attack_fn:   rfgsm, bim_attack, pgd_attack
    eps_list:    list of epsilons
    students:    list of adv-trained student models
    extra_kwargs_func: function(eps) -> dict of kwargs to pass to attack_fn
    """
    all_rows = []  # ["attack","epsilon","model","RMSE","run","seed"]

    hb_flag, hb_thread = start_heartbeat(f"{attack_name}_morphence_eval")
    start_all = time.time()

    try:
        for run_i in range(1, n_runs + 1):
            run_start = time.time()
            seed_i = 3000 + run_i
            random.seed(seed_i)
            np.random.seed(seed_i)
            torch.manual_seed(seed_i)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed_i)

            # clean
            base_clean   = eval_clean_rmse(base_eval, test_dl)
            morph_clean  = eval_ensemble_rmse(students, test_dl)
            all_rows.append(["clean", None, "base",           base_clean,  run_i, seed_i])
            all_rows.append(["clean", None, "morph_ensemble", morph_clean, run_i, seed_i])

            print(f"[{attack_name.upper()} | Run {run_i:02d}] clean | "
                  f"base={base_clean:.5f} | morph_ens={morph_clean:.5f}")

            # attack
            for eps in eps_list:
                atk_kwargs = extra_kwargs_func(eps)

                # base white box (attacker = defender = base_eval)
                base_rmse = eval_pair_attack(
                    defender=base_eval,
                    attacker=base_eval,
                    loader=test_dl,
                    attack_fn=attack_fn,
                    atk_kwargs=atk_kwargs,
                )

                # random switching
                # attacker+defender random from pool
                def atk_kwargs_wrapper():
                    return atk_kwargs

                morph_rmse = eval_random_switch_attack(
                    models=students,
                    loader=test_dl,
                    attack_fn=attack_fn,
                    atk_kwargs_func=atk_kwargs_wrapper,
                )

                all_rows.append([attack_name, eps, "base",       base_rmse,  run_i, seed_i])
                all_rows.append([attack_name, eps, "morph_rand", morph_rmse, run_i, seed_i])

                print(
                    f"[{attack_name.upper()} | Run {run_i:02d}] eps={eps:.2f} | "
                    f"base={base_rmse:.5f} | morph_rand={morph_rmse:.5f}"
                )

            # incremental save
            df_runs = pd.DataFrame(
                all_rows,
                columns=["attack","epsilon","model","RMSE","run","seed"]
            )
            df_runs.to_csv(results_csv, index=False)

            stats = (
                df_runs
                .groupby(["attack","epsilon","model"])["RMSE"]
                .agg(mean="mean", std="std", n="count")
                .reset_index()
                .sort_values(["attack","epsilon","model"])
            )
            stats.to_csv(stats_csv, index=False)

            run_mins   = (time.time() - run_start) / 60.0
            total_mins = (time.time() - start_all) / 60.0
            print(
                f"[{attack_name.upper()}] Completed run {run_i}/{n_runs} | "
                f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
                flush=True
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            gc.collect()

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    print(f"\nSaved {attack_name} runs to:", results_csv)
    print(f"Saved {attack_name} stats to:", stats_csv)
    return df_runs, stats

In [ ]:
# FGSM diversity + transferability + 30 run eval

print("\nFGSM diversity + transferability + 30 run eval")

def fgsm_kwargs(eps):
    return {"eps": eps, "alpha": 0.02}

_ = compute_weight_diversity(fgsm_students, fgsm_student_names, FGSM_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(fgsm_students, test_dl, FGSM_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=fgsm_students,
    names=fgsm_student_names,
    loader=test_dl,
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    extra_kwargs_func=fgsm_kwargs,
    out_csv=FGSM_XFER_MATRIX_CSV,
    attack_label="rfgsm",
)

df_fgsm_runs, fgsm_stats = run_morphence_eval(
    attack_name="rfgsm",
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    students=fgsm_students,
    results_csv=FGSM_RUNS_CSV,
    stats_csv=FGSM_STATS_CSV,
    extra_kwargs_func=fgsm_kwargs,
    n_runs=30,
)


FGSM diversity + transferability + 30 run eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_prediction_diversity.csv

[RFGSM] Transfer matrix for eps=0.1...
  atk=student_base_01_fgsm_adv - def=student_base_01_fgsm_adv | eps=0.10 | RMSE=0.04817
  atk=student_base_01_fgsm_adv - def=student_base_02_fgsm_adv | eps=0.10 | RMSE=0.04656
  atk=student_base_01_fgsm_adv - def=student_base_03_fgsm_adv | eps=0.10 | RMSE=0.04537
  atk=student_base_01_fgsm_adv - def=student_base_04_fgsm_adv | eps=0.10 | RMSE=0.04490
  atk=student_base_02_fgsm_adv - def=student_base_01_fgsm_adv | eps=0.10 | RMSE=0.04579
  atk=student_base_02_fgsm_adv - def=student_base_02_fgsm_adv | eps=0.10 | RMSE=0.04885
  atk=student_base_02_fgsm_adv - def=student_base_03_fgsm_adv | eps=0.10 | RMSE=0.04463
  atk=student_base_02_fgsm_adv - def=student_base_04_fgsm_adv | eps=0.10 | R

In [ ]:
# BIM diversity + transferability + 30 run eval

print("\nBIM diversity + transferability + 30 run eval")

def bim_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / BIM_ITERS,
        "iters": BIM_ITERS,
        "random_start": True,
    }

_ = compute_weight_diversity(bim_students, bim_student_names, BIM_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(bim_students, test_dl, BIM_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=bim_students,
    names=bim_student_names,
    loader=test_dl,
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=bim_kwargs,
    out_csv=BIM_XFER_MATRIX_CSV,
    attack_label="bim",
)

df_bim_runs, bim_stats = run_morphence_eval(
    attack_name="bim",
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    students=bim_students,
    results_csv=BIM_RUNS_CSV,
    stats_csv=BIM_STATS_CSV,
    extra_kwargs_func=bim_kwargs,
    n_runs=30,
)



BIM diversity + transferability + 30 run eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_prediction_diversity.csv

[BIM] Transfer matrix for eps=0.1...
  atk=student_base_01_bim_adv - def=student_base_01_bim_adv | eps=0.10 | RMSE=0.14616
  atk=student_base_01_bim_adv - def=student_base_02_bim_adv | eps=0.10 | RMSE=0.14124
  atk=student_base_01_bim_adv - def=student_base_03_bim_adv | eps=0.10 | RMSE=0.14404
  atk=student_base_01_bim_adv - def=student_base_04_bim_adv | eps=0.10 | RMSE=0.13979
  atk=student_base_02_bim_adv - def=student_base_01_bim_adv | eps=0.10 | RMSE=0.14616
  atk=student_base_02_bim_adv - def=student_base_02_bim_adv | eps=0.10 | RMSE=0.14124
  atk=student_base_02_bim_adv - def=student_base_03_bim_adv | eps=0.10 | RMSE=0.14404
  atk=student_base_02_bim_adv - def=student_base_04_bim_adv | eps=0.10 | RMSE=0.13979
  atk=stude

In [ ]:
# PGD diversity + transferability + 30 run eval

print("\nPGD diversity + transferability + 30 run eval")

def pgd_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / PGD_ITERS,
        "iters": PGD_ITERS,
        "random_start": True,
    }

_ = compute_weight_diversity(pgd_students, pgd_student_names, PGD_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(pgd_students, test_dl, PGD_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=pgd_students,
    names=pgd_student_names,
    loader=test_dl,
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=pgd_kwargs,
    out_csv=PGD_XFER_MATRIX_CSV,
    attack_label="pgd",
)

df_pgd_runs, pgd_stats = run_morphence_eval(
    attack_name="pgd",
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    students=pgd_students,
    results_csv=PGD_RUNS_CSV,
    stats_csv=PGD_STATS_CSV,
    extra_kwargs_func=pgd_kwargs,
    n_runs=30,
)


PGD diversity + transferability + 30 run eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_prediction_diversity.csv

[PGD] Transfer matrix for eps=0.1...
  atk=student_base_01_pgd_adv - def=student_base_01_pgd_adv | eps=0.10 | RMSE=0.11914
  atk=student_base_01_pgd_adv - def=student_base_02_pgd_adv | eps=0.10 | RMSE=0.08839
  atk=student_base_01_pgd_adv - def=student_base_03_pgd_adv | eps=0.10 | RMSE=0.08645
  atk=student_base_01_pgd_adv - def=student_base_04_pgd_adv | eps=0.10 | RMSE=0.08614
  atk=student_base_02_pgd_adv - def=student_base_01_pgd_adv | eps=0.10 | RMSE=0.09136
  atk=student_base_02_pgd_adv - def=student_base_02_pgd_adv | eps=0.10 | RMSE=0.11782
  atk=student_base_02_pgd_adv - def=student_base_03_pgd_adv | eps=0.10 | RMSE=0.08817
  atk=student_base_02_pgd_adv - def=student_base_04_pgd_adv | eps=0.10 | RMSE=0.09119
  atk=stude

In [ ]:
print("FGSM runs:", FGSM_RUNS_CSV)
print("BIM  runs:", BIM_RUNS_CSV)
print("PGD  runs:", PGD_RUNS_CSV)
print("FGSM weight diversity:", FGSM_WEIGHT_DIVERSITY_CSV)
print("BIM  weight diversity:", BIM_WEIGHT_DIVERSITY_CSV)
print("PGD  weight diversity:", PGD_WEIGHT_DIVERSITY_CSV)
print("FGSM transfer matrix:", FGSM_XFER_MATRIX_CSV)
print("BIM  transfer matrix:", BIM_XFER_MATRIX_CSV)
print("PGD  transfer matrix:", PGD_XFER_MATRIX_CSV)

FGSM runs: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_morphence_runs.csv
BIM  runs: /content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_morphence_runs.csv
PGD  runs: /content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_morphence_runs.csv
FGSM weight diversity: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_weight_diversity.csv
BIM  weight diversity: /content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_weight_diversity.csv
PGD  weight diversity: /content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_weight_diversity.csv
FGSM transfer matrix: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_transfer_matrix.csv
BIM  transfer matrix: /content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_transfer_matrix.csv
PGD  transfer matrix: /content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_transfer_matrix.csv


In [ ]:
import os
import pandas as pd
import numpy as np

# paths for runs, results, and extra artifacts

ATTACK_CONFIGS = [
    dict(
        name="FGSM",
        prefix="fgsm",
        attack_key="rfgsm",
        runs_csv="/content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_morphence_runs.csv",
        results_dir="/content/drive/MyDrive/298/Jena/proj_v2/fgsm/results",
        weight_diversity_csv="/content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_weight_diversity.csv",
        transfer_matrix_csv="/content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_transfer_matrix.csv",
    ),
    dict(
        name="BIM",
        prefix="bim",
        attack_key="bim",
        runs_csv="/content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_morphence_runs.csv",
        results_dir="/content/drive/MyDrive/298/Jena/proj_v2/bim/results",
        weight_diversity_csv="/content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_weight_diversity.csv",
        transfer_matrix_csv="/content/drive/MyDrive/298/Jena/proj_v2/bim/results/bim_transfer_matrix.csv",
    ),
    dict(
        name="PGD",
        prefix="pgd",
        attack_key="pgd",
        runs_csv="/content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_morphence_runs.csv",
        results_dir="/content/drive/MyDrive/298/Jena/proj_v2/pgd/results",
        weight_diversity_csv="/content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_weight_diversity.csv",
        transfer_matrix_csv="/content/drive/MyDrive/298/Jena/proj_v2/pgd/results/pgd_transfer_matrix.csv",
    ),
]


# compute and save stats for one attack config
def compute_stats_for_attack(cfg):
    name        = cfg["name"]
    prefix      = cfg["prefix"]
    attack_key  = cfg["attack_key"]
    runs_csv    = cfg["runs_csv"]
    results_dir = cfg["results_dir"]

    os.makedirs(results_dir, exist_ok=True)

    print(f"\n==============================================")
    print(f"Processing {name} runs")
    print(f"Runs CSV: {runs_csv}")
    print("==============================================")

    # load runs CSV
    df_runs = pd.read_csv(runs_csv)
    print("Loaded runs:", df_runs.shape)
    print(df_runs.head())

    # clean stats (base vs morph_ensemble)
    df_clean = df_runs[df_runs["attack"] == "clean"].copy()

    # ensure nice ordering of models
    df_clean["model"] = pd.Categorical(
        df_clean["model"],
        categories=["base", "morph_ensemble"],
        ordered=True
    )

    clean_stats = (
        df_clean
        .groupby("model", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("model")
    )

    print(f"\n[{name} | Clean] stats:")
    print(clean_stats.to_string(index=False))

    clean_csv = os.path.join(results_dir, f"{prefix}_clean_stats.csv")
    clean_stats.to_csv(clean_csv, index=False)
    print("Saved clean stats to:", clean_csv)

    # attack stats (base vs morph_rand) per epsilon
    df_attack = df_runs[df_runs["attack"] == attack_key].copy()

    def model_attack_stats(df, model_name):
        st = (
            df[df["model"] == model_name]
            .groupby("epsilon", observed=False)["RMSE"]
            .agg(
                mean   ="mean",
                std    ="std",
                min    ="min",
                q1     =lambda s: s.quantile(0.25),
                median ="median",
                q3     =lambda s: s.quantile(0.75),
                max    ="max",
                n      ="count",
            )
            .reset_index()
            .sort_values("epsilon")
        )

        print(f"\n[{name} | {attack_key}] {model_name} stats:")
        print(st.to_string(index=False))

        out_csv = os.path.join(results_dir, f"{prefix}_{attack_key}_{model_name}_stats.csv")
        st.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}")

    # base white-box stats
    model_attack_stats(df_attack, "base")

    # morphence
    # random-switch ensemble under attack
    model_attack_stats(df_attack, "morph_rand")



# Run for all three: FGSM, BIM, PGD
for cfg in ATTACK_CONFIGS:
    compute_stats_for_attack(cfg)



Processing FGSM runs
Runs CSV: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_morphence_runs.csv
Loaded runs: (300, 6)
  attack  epsilon           model      RMSE  run  seed
0  clean      NaN            base  0.014594    1  3001
1  clean      NaN  morph_ensemble  0.016413    1  3001
2  rfgsm      0.1            base  0.072543    1  3001
3  rfgsm      0.1      morph_rand  0.046531    1  3001
4  rfgsm      0.2            base  0.097718    1  3001

[FGSM | Clean] stats:
         model     mean  std      min       q1   median       q3      max  n
          base 0.014594  0.0 0.014594 0.014594 0.014594 0.014594 0.014594 30
morph_ensemble 0.016413  0.0 0.016413 0.016413 0.016413 0.016413 0.016413 30
Saved clean stats to: /content/drive/MyDrive/298/Jena/proj_v2/fgsm/results/fgsm_clean_stats.csv

[FGSM | rfgsm] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.072646 0.000231 0.072115 0.072496 0.072629 0.072816 0.073047 30
     0.2 0.